# Markov Random Field (MRF) — Visual Learning Notebook

This version is built for **learning by seeing**. Every concept has a plot.

You will learn:
1. What nodes, edges, and potentials *look* like
2. How Gibbs sampling walks through states
3. How a grid MRF denoises an image step-by-step
4. Energy, marginals, and convergence visually

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import cm
from itertools import product
import matplotlib.patches as mpatches

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120

## 1. Core MRF with visualization helpers

In [ ]:
class MarkovNetwork:
    def __init__(self):
        self.graph = nx.Graph()
        self.node_potentials = {}
        self.edge_potentials = {}

    def add_node(self, node, states, potential=None):
        self.graph.add_node(node)
        self.graph.nodes[node]['states'] = list(states)
        if potential is None: potential = np.ones(len(states))
        self.node_potentials[node] = np.array(potential, dtype=float)

    def add_edge(self, u, v, potential=None):
        self.graph.add_edge(u,v)
        su, sv = len(self.graph.nodes[u]['states']), len(self.graph.nodes[v]['states'])
        if potential is None: potential = np.ones((su,sv))
        pot = np.array(potential, dtype=float)
        self.edge_potentials[(u,v)] = pot
        self.edge_potentials[(v,u)] = pot.T

    def draw(self, title="MRF Structure"):
        plt.figure(figsize=(4,3))
        pos = nx.spring_layout(self.graph, seed=1)
        nx.draw(self.graph, pos, with_labels=True, node_color='#A0C4FF', 
                node_size=1200, edge_color='gray', width=2)
        plt.title(title); plt.axis('off'); plt.show()

    def plot_potentials(self):
        n = len(self.graph.nodes)
        fig, axes = plt.subplots(1, n+len(self.graph.edges), figsize=(3*(n+len(self.graph.edges)),3))
        if n+len(self.graph.edges)==1: axes=[axes]
        idx=0
        for node in self.graph.nodes:
            ax=axes[idx]; pot=self.node_potentials[node]
            ax.bar(self.graph.nodes[node]['states'], pot, color='#FFB5A7')
            ax.set_title(f'φ_{node}(x)') ; ax.set_ylim(0, pot.max()*1.2); idx+=1
        for u,v in self.graph.edges:
            ax=axes[idx]; pot=self.edge_potentials[(u,v)]
            im=ax.imshow(pot, cmap='viridis', vmin=0)
            ax.set_xticks(range(len(self.graph.nodes[v]['states'])))
            ax.set_yticks(range(len(self.graph.nodes[u]['states'])))
            ax.set_xticklabels(self.graph.nodes[v]['states'])
            ax.set_yticklabels(self.graph.nodes[u]['states'])
            ax.set_xlabel(f'{v}'); ax.set_ylabel(f'{u}')
            ax.set_title(f'φ_{u}{v}'); plt.colorbar(im, ax=ax, fraction=0.046); idx+=1
        plt.tight_layout(); plt.show()

    def gibbs_sample(self, n_iter=1000, burn_in=200, track_energy=False):
        nodes=list(self.graph.nodes)
        state={n: np.random.choice(self.graph.nodes[n]['states']) for n in nodes}
        samples=[]; energies=[]
        for t in range(n_iter):
            for n in nodes:
                states=self.graph.nodes[n]['states']
                probs=[]
                for s in states:
                    p=self.node_potentials[n][states.index(s)]
                    for nb in self.graph.neighbors(n):
                        iu=states.index(s); iv=self.graph.nodes[nb]['states'].index(state[nb])
                        p*=self.edge_potentials[(n,nb)][iu,iv]
                    probs.append(p)
                probs=np.array(probs); probs/=probs.sum()
                state[n]=np.random.choice(states, p=probs)
            if track_energy:
                e=-np.log(self.joint_probability(state)+1e-12)
                energies.append(e)
            if t>=burn_in: samples.append(state.copy())
        return samples, energies

    def joint_probability(self, assignment):
        prob=1.0
        for n in self.graph.nodes:
            idx=self.graph.nodes[n]['states'].index(assignment[n])
            prob*=self.node_potentials[n][idx]
        for u,v in self.graph.edges:
            iu=self.graph.nodes[u]['states'].index(assignment[u])
            iv=self.graph.nodes[v]['states'].index(assignment[v])
            prob*=self.edge_potentials[(u,v)][iu,iv]
        return prob

## 2. Build a tiny 3-node chain A—B—C
Visualize structure and potentials.

In [ ]:
mrf = MarkovNetwork()
for node in ['A','B','C']:
    mrf.add_node(node, states=[0,1], potential=[1,1])  # uniform prior

# Edge prefers equality: high when same, low when different
edge_pot = np.array([[3.0, 1.0],
                     [1.0, 3.0]])
mrf.add_edge('A','B', edge_pot)
mrf.add_edge('B','C', edge_pot)

mrf.draw('3-node Markov Chain')
mrf.plot_potentials()

Interpretation: the heatmaps show φ_AB and φ_BC. Dark purple = low, yellow = high. High values on diagonal mean neighbors like to be equal.

## 3. Exact joint distribution (enumerate all 2³=8 states)

In [ ]:
states = list(product([0,1], repeat=3))
probs=[]; labels=[]
for a,b,c in states:
    assign={'A':a,'B':b,'C':c}
    p=mrf.joint_probability(assign)
    probs.append(p); labels.append(f"{a}{b}{c}")
probs=np.array(probs); probs/=probs.sum()

plt.figure(figsize=(7,3))
plt.bar(labels, probs, color='#90BE6D')
plt.title('Exact Joint P(A,B,C) — notice 000 and 111 dominate')
plt.ylabel('Probability'); plt.xlabel('Configuration'); plt.show()

for lab,p in zip(labels,probs): print(lab, f"{p:.3f}")

## 4. Gibbs sampling walk — watch it live

In [ ]:
samples, energies = mrf.gibbs_sample(n_iter=500, burn_in=0, track_energy=True)

# Trace plot
trace = np.array([[s['A'], s['B'], s['C']] for s in samples])
plt.figure(figsize=(10,3))
for i,lab in enumerate(['A','B','C']):
    plt.plot(trace[:,i]+i*1.5, label=lab, drawstyle='steps-mid')
plt.yticks([0,0.5,1,1.5,2,2.5,3],[0,1,'',0,1,'',0,1][:7])
plt.title('Gibbs Sampling Trace (each line is a node)')
plt.xlabel('Iteration'); plt.legend(); plt.show()

# Energy plot
plt.figure(figsize=(6,3))
plt.plot(energies, color='crimson')
plt.title('Energy -log P(x) over time — drops as sampler finds high-prob states')
plt.xlabel('Iteration'); plt.ylabel('Energy'); plt.show()

## 5. Marginals converge

In [ ]:
burn=100
samples_post = samples[burn:]
marginals={n: np.mean([s[n]==1 for s in samples_post]) for n in ['A','B','C']}

plt.bar(marginals.keys(), marginals.values(), color='#577590')
plt.ylim(0,1); plt.title('Estimated P(node=1) after burn-in'); plt.ylabel('Probability')
for k,v in marginals.items(): plt.text(k, v+0.02, f"{v:.2f}", ha='center')
plt.show()

## 6. Grid MRF — Image Denoising Visualized
Each pixel is a node connected to 4 neighbors. We visualize the process.

In [ ]:
def build_grid_mrf(observed, beta=1.0, eta=2.0):
    h,w=observed.shape; m=MarkovNetwork()
    for i in range(h):
        for j in range(w):
            obs=observed[i,j]
            pot=[np.exp(-eta*obs), np.exp(eta*obs)]  # states [-1,1]
            m.add_node((i,j), states=[-1,1], potential=pot)
    edge=np.array([[np.exp(beta), np.exp(-beta)],[np.exp(-beta), np.exp(beta)]])
    for i in range(h):
        for j in range(w):
            if i+1<h: m.add_edge((i,j),(i+1,j), edge)
            if j+1<w: m.add_edge((i,j),(i,j+1), edge)
    return m

# Create synthetic image
true = np.ones((30,30)); true[8:22,8:22] = -1; true[12:18,12:18]=1
noisy = true * np.random.choice([-1,1], size=true.shape, p=[0.15,0.85])

fig,ax=plt.subplots(1,2,figsize=(6,3))
ax[0].imshow(true, cmap='gray', vmin=-1, vmax=1); ax[0].set_title('True'); ax[0].axis('off')
ax[1].imshow(noisy, cmap='gray', vmin=-1, vmax=1); ax[1].set_title('Noisy Observation'); ax[1].axis('off')
plt.show()

In [ ]:
mrf_grid = build_grid_mrf(noisy, beta=0.9, eta=1.8)

# Run Gibbs and store snapshots
nodes=list(mrf_grid.graph.nodes)
state={n: np.random.choice([-1,1]) for n in nodes}
snapshots=[]
for t in range(120):
    for n in nodes:
        probs=[]
        for s in [-1,1]:
            p=mrf_grid.node_potentials[n][0 if s==-1 else 1]
            for nb in mrf_grid.graph.neighbors(n):
                iu=0 if s==-1 else 1; iv=0 if state[nb]==-1 else 1
                p*=mrf_grid.edge_potentials[(n,nb)][iu,iv]
            probs.append(p)
        probs=np.array(probs); probs/=probs.sum()
        state[n]=np.random.choice([-1,1], p=probs)
    if t%20==0: snapshots.append(np.array([[state[(i,j)] for j in range(30)] for i in range(30)]))

fig, axes=plt.subplots(2,3, figsize=(9,6))
axes=axes.ravel()
for i,img in enumerate(snapshots):
    axes[i].imshow(img, cmap='gray', vmin=-1, vmax=1)
    axes[i].set_title(f'Iter {i*20}'); axes[i].axis('off')
plt.suptitle('MRF Denoising Evolution — watch noise disappear', y=0.95)
plt.tight_layout(); plt.show()

## 7. Visualize Markov Blanket
Pick a center pixel — its probability depends only on neighbors.

In [ ]:
center=(15,15)
neighbors=list(mrf_grid.graph.neighbors(center))

grid_vis = np.zeros((30,30,3))
grid_vis[:,:,0]=0.9; grid_vis[:,:,1]=0.9; grid_vis[:,:,2]=0.9  # light gray
for n in neighbors: grid_vis[n[0],n[1]]=[1,0.6,0.2]  # orange neighbors
grid_vis[center[0],center[1]]=[0.2,0.4,0.9]  # blue center

plt.figure(figsize=(4,4))
plt.imshow(grid_vis, interpolation='nearest')
plt.title('Markov Blanket: center depends only on 4 orange neighbors')
plt.axis('off')
blue = mpatches.Patch(color=[0.2,0.4,0.9], label='Target pixel')
orange = mpatches.Patch(color=[1,0.6,0.2], label='Markov blanket')
plt.legend(handles=[blue, orange], loc='upper right', bbox_to_anchor=(1.3,1))
plt.show()

## 8. What to play with
- Change `beta` (0.2 → very noisy, 2.0 → oversmooth)
- Change `eta` (trust observation vs neighbors)
- Try different graph structures

This notebook gives you visual intuition before math. Next step: belief propagation visualization?